# 🥇 Camada Gold — Modelos Analíticos (Star Schema)

## O que é a camada Gold?
A Gold é a camada **orientada ao consumo**. Os dados aqui estão prontos para:
- Dashboards e ferramentas de BI (Power BI, Looker, Metabase)
- Consultas analíticas de alta performance
- Relatórios executivos e KPIs

## Por que usar Star Schema?
O **Star Schema** separa os dados em:
- **Tabela Fato**: registros de eventos/transações com métricas numéricas
- **Tabelas Dimensão**: contexto descritivo (quem, o quê, onde, quando)

Isso **elimina redundância**, **acelera joins** e facilita a compreensão por analistas de negócio.

```
                 ┌─────────────────┐
                 │  DIM_MATERIAL   │
                 │  id_material    │
                 │  desc_material  │
                 └────────┬────────┘
                          │
┌─────────────────┐       │       ┌─────────────────┐
│   DIM_CENTRO    │       │       │  DIM_TIPO_ESTOQUE│
│  id_centro      ├───────┤───────┤  id_tipo         │
│  desc_centro    │       │       │  desc_tipo        │
└─────────────────┘  FATO │       └─────────────────┘
                    ESTOQUE│
                 ┌─────────┴───────┐
                 │  id_material FK │
                 │  id_centro FK   │
                 │  id_tipo FK     │
                 │  qtd_estoque    │  ← MÉTRICA
                 │  data_referencia│
                 └─────────────────┘
```

In [ ]:
# ── Configuração para Google Colab ────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/meu-projeto-medalhao')
# !pip install -q pandas pyarrow

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

SILVER = Path('../data/silver')
GOLD   = Path('../data/gold')
GOLD.mkdir(parents=True, exist_ok=True)

print('Ambiente pronto.')

## 1. Carregar dados da Silver

In [ ]:
df = pd.read_parquet(SILVER / 'estoque_materiais.parquet')

print(f'Silver carregado: {df.shape}')
display(df[['NATB', 'MAKTX', 'WERKS', 'MAINS', 'LABST']].head(10))

## 2. Construir Dimensões

### DIM_MATERIAL
Contém o catálogo único de materiais. Uma linha por material.

In [ ]:
dim_material = (
    df[['NATB', 'MAKTX']]
    .drop_duplicates(subset=['NATB'])
    .rename(columns={'NATB': 'id_material', 'MAKTX': 'desc_material'})
    .sort_values('id_material')
    .reset_index(drop=True)
)

print(f'DIM_MATERIAL: {len(dim_material)} materiais únicos')
display(dim_material)

### DIM_CENTRO
Representa os centros/plantas. Em um cenário real, viria de uma tabela mestre do ERP.

In [ ]:
# Mapeamento de códigos → descrições (simulando tabela mestre do ERP)
centro_descricoes = {
    'BT10': 'Armazém Central Recife',
    'BT50': 'Centro de Distribuição Norte',
}

dim_centro = (
    df[['WERKS']]
    .drop_duplicates()
    .rename(columns={'WERKS': 'id_centro'})
    .assign(desc_centro=lambda x: x['id_centro'].map(centro_descricoes))
    .sort_values('id_centro')
    .reset_index(drop=True)
)

print(f'DIM_CENTRO: {len(dim_centro)} centros únicos')
display(dim_centro)

### DIM_TIPO_ESTOQUE
Dimensão do tipo de estoque (MAINS).

In [ ]:
tipo_descricoes = {
    100: 'Estoque Livre (Unrestricted)',
    200: 'Em Controle de Qualidade',
    300: 'Bloqueado',
}

dim_tipo_estoque = (
    df[['MAINS']]
    .drop_duplicates()
    .rename(columns={'MAINS': 'id_tipo'})
    .assign(desc_tipo=lambda x: x['id_tipo'].map(tipo_descricoes))
    .sort_values('id_tipo')
    .reset_index(drop=True)
)

print(f'DIM_TIPO_ESTOQUE: {len(dim_tipo_estoque)} tipos únicos')
display(dim_tipo_estoque)

## 3. Construir Tabela Fato

A **FATO_ESTOQUE** conecta todas as dimensões e contém a métrica central: `qtd_estoque`.

> **Regra do Star Schema:** A Fato deve conter apenas **chaves estrangeiras** (FK) e **métricas**.
> Nenhum atributo descritivo deve estar na Fato — isso causaria redundância (conflito com as dimensões).

In [ ]:
fato_estoque = (
    df[['NATB', 'WERKS', 'MAINS', 'LABST']]
    .rename(columns={
        'NATB'  : 'id_material',   # FK → DIM_MATERIAL
        'WERKS' : 'id_centro',     # FK → DIM_CENTRO
        'MAINS' : 'id_tipo',       # FK → DIM_TIPO_ESTOQUE
        'LABST' : 'qtd_estoque',   # MÉTRICA
    })
    .assign(
        data_referencia=date.today().isoformat(),  # data de referência da snapshot
        sk_estoque=lambda x: range(1, len(x)+1),   # surrogate key
    )
    # surrogate key como primeiro campo (boa prática de DW)
    [['sk_estoque', 'id_material', 'id_centro', 'id_tipo', 'qtd_estoque', 'data_referencia']]
)

print(f'FATO_ESTOQUE: {len(fato_estoque)} registros')
display(fato_estoque)

## 4. Tabela Analítica — Agregação por Centro
Além do Star Schema, criamos uma **tabela wide** pré-agregada para uso direto em BI.
Esta é a tabela que o analista vai usar no Power BI sem precisar escrever SQL.

In [ ]:
# JOIN entre Fato e Dimensões para gerar visão desnormalizada
gold_analitica = (
    fato_estoque
    .merge(dim_material,     on='id_material', how='left')
    .merge(dim_centro,       on='id_centro',   how='left')
    .merge(dim_tipo_estoque, on='id_tipo',      how='left')
)

print('=== Visão analítica completa (Gold) ===')
display(gold_analitica)

# Agregação por centro: estoque total e quantidade de itens distintos
gold_resumo_centro = (
    gold_analitica
    .groupby(['id_centro', 'desc_centro'], as_index=False)
    .agg(
        total_itens      =('id_material', 'count'),
        itens_distintos  =('id_material', 'nunique'),
        estoque_total    =('qtd_estoque', 'sum'),
        estoque_medio    =('qtd_estoque', 'mean'),
        estoque_maximo   =('qtd_estoque', 'max'),
    )
    .round({'estoque_medio': 2})
)

print('\n=== KPI: Estoque por Centro ===')
display(gold_resumo_centro)

## 5. Persistir na Gold

In [ ]:
def save_gold(df: pd.DataFrame, name: str):
    """Salva em Parquet (principal) e CSV (conveniência para BI)."""
    p = GOLD / f'{name}.parquet'
    c = GOLD / f'{name}.csv'
    df.to_parquet(p, index=False)
    df.to_csv(c, index=False, sep=';')
    print(f'[Gold] ✓ {name} → Parquet ({p.stat().st_size}b) + CSV ({c.stat().st_size}b)')

save_gold(dim_material,     'dim_material')
save_gold(dim_centro,       'dim_centro')
save_gold(dim_tipo_estoque, 'dim_tipo_estoque')
save_gold(fato_estoque,     'fato_estoque')
save_gold(gold_analitica,   'gold_visao_analitica')
save_gold(gold_resumo_centro, 'gold_kpi_por_centro')

## 6. Head() — Visualização final da Gold
> **O que mudou?** Os dados estão em modelo relacional otimizado (Star Schema).
> A Fato contém apenas métricas e FKs. As Dimensões são reutilizáveis em qualquer análise.
> A visão analítica desnormalizada está pronta para BI sem joins adicionais.

In [ ]:
print('=== HEAD — FATO_ESTOQUE ===')
display(fato_estoque.head())

print('\n=== HEAD — DIM_MATERIAL ===')
display(dim_material.head())

print('\n=== HEAD — DIM_CENTRO ===')
display(dim_centro.head())

print('\n=== HEAD — KPI por Centro (pronto para BI) ===')
display(gold_resumo_centro.head())

print('\n=== Arquivos gerados na Gold ===')
for f in sorted(GOLD.iterdir()):
    print(f'  {f.name:40s}  {f.stat().st_size:>8} bytes')

## Resumo Final — Arquitetura Medallion

| Camada | Responsabilidade | Formato | Regra Principal |
|---|---|---|---|
| **Bronze** | Preservar o dado bruto | Parquet | **Nunca transformar o conteúdo** |
| **Silver** | Limpar e padronizar | Parquet | Tipos corretos, sem duplicatas |
| **Gold** | Modelar para consumo | Parquet + CSV | Otimizado para leitura e BI |

### Tabelas Gold geradas
| Tabela | Tipo | Descrição |
|---|---|---|
| `fato_estoque` | Fato | Posição de estoque por material/centro |
| `dim_material` | Dimensão | Catálogo de materiais |
| `dim_centro` | Dimensão | Plantas/armazéns |
| `dim_tipo_estoque` | Dimensão | Tipos de estoque |
| `gold_visao_analitica` | Wide/Flat | Join completo para análise ad-hoc |
| `gold_kpi_por_centro` | Agregada | KPIs de estoque por centro (para dashboard) |